# Fig. 4i–k — one PC network through five environments in sequence

A single prospective-configuration (PC) network is run through **five environment blocks back-to-back**
(20 000 timesteps each), carrying its weights across block boundaries. We track, over the whole
run:

* **P(choice a)** — median across seeds, shaded IQR band; shows the network transitioning as the
  contingency changes block to block.
* **run-length verdict** — one binary call per block: over the whole block, are the choice
  run-lengths better described by a **power law** (heavy-tailed foraging) or an **exponential**
  (regular switching)? Fit with the CSN protocol (`ssb.analysis.powerlaw_fit`).

For **each block** we sample trajectories from a few seeds and draw the emergent attractor, both in
**2D** ($w_2-w_1$ vs $x_2-x_1$) and in **3D** ($x_0$, $w_2-w_1$, $x_2-x_1$, the PM-attractor axes).

The blocks:

Arm labels follow Fig. 4i: arm **a** is action 1 (output unit 2) and arm **b** is action 0 (output unit 1).

1. deterministic, $s=1$ if choice **a**, $s=0$ if choice **b**
2. deterministic, $s=0$ if choice **a**, $s=0.4$ if choice **b**
3. unbaited bandit, $P=0.15$ for arm **a**, $0.45$ for arm **b**, ITI $=1$ between trials
4. $s=1$ regardless of choice, **no feedback** (the decay condition from the forgetting notebook)
5. deterministic, $s=0$ if choice **a**, $s=0.4$ if choice **b** (same as block 2 — relearning)

In [ ]:
# ---- imports & path -----------------------------------------------------------------
import os, sys, time
import numpy as np
import jax, jax.numpy as jnp
from jax import vmap
import matplotlib as mpl, matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401  (registers 3d projection)
jax.config.update('jax_platform_name', 'cpu')

from ssb.models import ssb, rules
from ssb.utils.batch_inits import make_default_hps, batchify_hps, init_params_batch
from ssb.envs.bandits import (bandit_det, bandit_unbait_iti,
                              make_iti_state, make_iti_env_par)
from ssb.analysis.powerlaw_fit import action_lengths, fit_and_compare

try:
    mpl.rcParams['font.family'] = 'Arial'
except Exception:
    pass
mpl.rcParams['svg.fonttype'] = 'none'

# ---- plotting helpers ----
def nature_ax(ax, xlabel='', ylabel='', nticks=4, nx=None, ny=None):
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'): ax.spines[sp].set_linewidth(0.25)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nx or nticks))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(ny or nticks))
    ax.tick_params(labelsize=7, width=0.25)
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)

def side_legend(ax, **kw):
    ax.legend(fontsize=7, frameon=False, loc='upper left', bbox_to_anchor=(1.02, 1.0), **kw)

# ---- run-length utility (from F5) ----
def run_lengths(choices):
    "Lengths of maximal constant-choice runs in a 0/1 sequence (each >= 1)."
    return action_lengths(np.asarray(choices))

print('jax', jax.__version__, '| ssb loaded')

In [ ]:
# ---- CONFIG — all knobs here. Edit, then Run All. -----------------------------------
RULE   = 'PC'
SAVE   = True
FIGDIR = 'svgs'; os.makedirs(FIGDIR, exist_ok=True)

# PC hyperparameters — the exploration regime used in F5 (so foraging structure can appear
# in the bandit block). alpha=activity rate, beta=weight rate, eta_a=activity/escape noise.
ALPHA      = 0.10
BETA      = 0.01
ETA_A      = 0.005
INIT_SCALE = 0.25                  # commitment depth of the initial weights
CLIP       = 3.0                   # activity clip (matches F5 / F4_explore)
SEED       = 0

# ---- the five environment blocks, run in order, weights carried across ----
#   kind 'det' : bandit_det, env_par = reward vector [reward|b, reward|a]  (indexed by action: 0 = b, 1 = a)
#   kind 'iti' : bandit_unbait_iti, env_par = [prob|b, prob|a, iti]
# arm a = action 1 (output unit 2), arm b = action 0 (output unit 1), as labelled in Fig. 4i.
# choice a  <=>  x2-x1 >= 0 (ties counted as a).
BLOCKS = [
    dict(name='1. det  a:1 / b:0',      kind='det', par=(0.0, 1.0),  iti=0),
    dict(name='2. det  a:0 / b:0.4',    kind='det', par=(0.4, 0.0),  iti=0),
    dict(name='3. bandit a:.15 / b:.45 ITI1', kind='iti', par=(0.45, 0.15), iti=1),
    dict(name='4. s=1, no feedback',    kind='det', par=(1.0, 1.0),  iti=0),
    dict(name='5. det  a:0 / b:0.4',    kind='det', par=(0.4, 0.0),  iti=0),
]

# ---- run scale ----------------------------------------------------------------------
SCALE = 'full'                  # 'reduced' for a quick test, 'full' for the figure
if SCALE == 'reduced':
    N_SEEDS = 12
    TS      = 4_000                # timesteps per block
else:  # 'full'
    N_SEEDS = 48
    TS      = 20_000

# ---- analysis knobs ----
ROLL   = 400                       # boxcar window for the P(choice a) trace (per seed)
PL_MIN_RUNS = 8                    # need at least this many pooled runs to attempt a block fit
ATTR_MAX_SEEDS = 6                 # how many seeds to sample for the attractor clouds
ATTR_N_SHOW    = 3000              # points drawn per attractor panel

NB = len(BLOCKS); TOTAL = NB * TS
print(f'scale={SCALE}: {N_SEEDS} seeds x {NB} blocks x {TS} steps = {TOTAL} steps total')

In [ ]:
# ---- simulation core: one PC net through the block sequence, state carried across ----
# Mirrors F5's per-step update (env -> stimulus -> settle -> noise/clip -> weights -> states)
# and logs, every step, the attractor coordinates used throughout the paper:
#   x0 = input activity, dW = w2-w1 (line-attractor / marginal axis), dX = x2-x1 (motor diff).
# choice a (action 1) <=> dX >= 0 (ties counted as a).  is_bandit flags real decision steps (False on ITI nulls).

def _base_hps():
    hps = make_default_hps(SEED)                 # sizes [[1,0],[2,0]] : 1 input, 2 motor
    hps['init_scale'] = jnp.array([INIT_SCALE])
    hps['alpha'] = jnp.array([ALPHA]); hps['beta'] = jnp.array([BETA])
    hps['eta_a'] = jnp.array([ETA_A])
    return batchify_hps(hps, N_SEEDS)

def _make_step(env_fn, hb, is_iti):
    uw, us = rules.rule2fns(RULE)
    UA = vmap(ssb.update_acts); UW = vmap(uw); US = vmap(us)
    NO = vmap(ssb.act_noise);   EX = vmap(ssb.expand_stimulus)
    envb = vmap(env_fn, in_axes=(0, 0, 0, 0))
    n = N_SEEDS
    @jax.jit
    def step(carry, _):
        acts, states, weights, es, key, EP, tern = carry
        is_bandit = (es[:, 2] == 0) if is_iti else jnp.ones(n, dtype=bool)
        rb, action, nes, key = envb(es, acts[:, -2:], EP, key)
        stim = EX(rb, acts)
        acts = UA(stim, acts, states, weights, tern, hb)
        acts, key = NO(acts, key, hb); acts = jnp.clip(acts, 0., CLIP)
        weights = UW(stim, acts, states, weights, tern, hb)
        states = US(acts, states, hb)
        x0 = acts[:, 0]
        dW = weights[:, 0, -1] - weights[:, 0, -2]
        dX = acts[:, -1] - acts[:, -2]
        return (acts, states, weights, nes, key, EP, tern), (dX >= 0, is_bandit, x0, dW, dX)
    return step

def run_sequence():
    hb = _base_hps()
    acts, states, weights, tern, key = init_params_batch(hb)
    choice = np.zeros((TOTAL, N_SEEDS), bool)
    bandit = np.zeros((TOTAL, N_SEEDS), bool)
    X0 = np.zeros((TOTAL, N_SEEDS), np.float32)
    DW = np.zeros((TOTAL, N_SEEDS), np.float32)
    DX = np.zeros((TOTAL, N_SEEDS), np.float32)
    t0 = time.time()
    for bi, blk in enumerate(BLOCKS):
        if blk['kind'] == 'iti':
            env_fn = bandit_unbait_iti
            EP = jnp.tile(make_iti_env_par(blk['par'][0], blk['par'][1], blk['iti'])[None], (N_SEEDS, 1))
            es = jnp.tile(make_iti_state(blk['iti'])[None], (N_SEEDS, 1))
            is_iti = True
        else:
            env_fn = bandit_det
            EP = jnp.tile(jnp.array(blk['par'])[None], (N_SEEDS, 1))
            es = jnp.zeros((N_SEEDS, 2))
            is_iti = False
        step = _make_step(env_fn, hb, is_iti)
        carry = (acts, states, weights, es, key, EP, tern)
        carry, out = jax.lax.scan(step, carry, None, length=TS)
        acts, states, weights, es, key, EP, tern = carry
        sl = slice(bi * TS, (bi + 1) * TS)
        choice[sl] = np.asarray(out[0]); bandit[sl] = np.asarray(out[1])
        X0[sl] = np.asarray(out[2]); DW[sl] = np.asarray(out[3]); DX[sl] = np.asarray(out[4])
        print(f"  block {blk['name']:28s} done  ({time.time()-t0:5.1f}s)  "
              f"P(a) end={choice[sl][-500:].mean():.2f}")
    print('total', round(time.time() - t0, 1), 's')
    return dict(choice=choice, bandit=bandit, X0=X0, DW=DW, DX=DX)

In [ ]:
# ---- run (cached to npz so figure-only reruns are instant) --------------------------
_CACHE = f'f6_data_{SCALE}_{N_SEEDS}x{TS}_choice_a.npz'   # 'choice' = chose arm a (action 1)
if os.path.exists(_CACHE):
    z = np.load(_CACHE)
    S = {k: z[k] for k in ('choice', 'bandit', 'X0', 'DW', 'DX')}
    print('loaded', _CACHE)
else:
    S = run_sequence()
    np.savez_compressed(_CACHE, **S)
    print('saved', _CACHE)
choice, bandit, X0, DW, DX = S['choice'], S['bandit'], S['X0'], S['DW'], S['DX']

In [ ]:
# ---- P(choice a) over time: per-seed boxcar, then median + IQR band across seeds ----
def _boxcar(x, k):
    return x if k <= 1 else np.convolve(x, np.ones(k) / k, mode='same')

pA_seed = np.stack([_boxcar(choice[:, s].astype(float), ROLL) for s in range(N_SEEDS)], axis=1)
pA_med = np.median(pA_seed, axis=1)
pA_lo  = np.percentile(pA_seed, 25, axis=1)
pA_hi  = np.percentile(pA_seed, 75, axis=1)

# ---- run-length verdict PER BLOCK: power law (1) vs exponential/other (0) --------------
# One CSN fit over the WHOLE block: pool the choice run-lengths across all seeds within the
# block and run the power-law-vs-exponential comparison (fixed xmin=1, the stable full-support
# discriminator used in F5). More runs per fit than a windowed version -> a firmer verdict.
def block_verdict(bi):
    lo, hi = bi * TS, (bi + 1) * TS
    runs = []
    for s in range(N_SEEDS):
        seg = choice[lo:hi, s]
        if bandit[lo:hi, s].any() and not bandit[lo:hi, s].all():
            seg = seg[bandit[lo:hi, s]]          # ITI block: decisions only
        rl = run_lengths(seg)
        if rl.size: runs.append(rl)
    runs = np.concatenate(runs) if runs else np.array([], int)
    if runs.size < PL_MIN_RUNS or np.unique(runs).size < 2:
        return np.nan, runs.size, None
    try:
        fr = fit_and_compare(runs, xmin=1)
        return (1.0 if fr.favored == 'power_law' else 0.0), runs.size, fr
    except Exception:
        return np.nan, runs.size, None

block_fits = [block_verdict(bi) for bi in range(NB)]
block_verd = np.array([v for v, _, _ in block_fits])
for bi, (v, n, fr) in enumerate(block_fits):
    tag = 'n/a' if np.isnan(v) else ('power law' if v > 0.5 else 'exponential')
    extra = '' if fr is None else f'  (alpha={fr.alpha:.2f}, p={fr.loglik_ratio_p:.3f})'
    print(f"  block {bi+1}: {tag:11s}  [{n} runs]{extra}")

In [ ]:
# ---- Figure A: time series (verdict strip + P(choice a)) ----------------------------
BLOCK_SHADE = ['#f3efe9', '#eef2f6', '#eef5ee', '#f6eef3', '#eef2f6']   # faint per-block tint
CH_A = '#8B6BA8'                                                        # choice-a colour
PL_C, EXP_C, NA_C = '#2e7d32', '#b0632a', '#dddddd'                     # power law / exp / n.a.

fig, (axv, axp) = plt.subplots(2, 1, figsize=(6.4, 2.4), sharex=True,
                               gridspec_kw={'height_ratios': [0.16, 1.0], 'hspace': 0.08})

# verdict strip — one verdict per block (fit over the whole block)
for bi, v in enumerate(block_verd):
    c = NA_C if np.isnan(v) else (PL_C if v > 0.5 else EXP_C)
    axv.axvspan(bi * TS, (bi + 1) * TS, color=c, lw=0)
    if bi: axv.axvline(bi * TS, color='white', lw=0.8)
axv.set_yticks([]); axv.set_xlim(0, TOTAL)
for sp in ('top', 'right', 'left', 'bottom'): axv.spines[sp].set_visible(False)
axv.set_ylabel('runs', fontsize=7, rotation=0, ha='right', va='center')
axv.text(0.0, 1.35, 'run-length verdict:', transform=axv.transAxes, fontsize=7, va='bottom')
axv.legend(handles=[Patch(color=PL_C, label='power law'), Patch(color=EXP_C, label='exponential'),
                    Patch(color=NA_C, label='n/a')],
           fontsize=6, frameon=False, ncol=3, loc='lower left', bbox_to_anchor=(0.24, 1.0),
           handlelength=1.0, columnspacing=1.0)

# P(choice a)
t = np.arange(0, TOTAL, 100)
for bi in range(NB):
    axp.axvspan(bi * TS, (bi + 1) * TS, color=BLOCK_SHADE[bi], lw=0, zorder=0)
    if bi: axp.axvline(bi * TS, color='#cccccc', lw=0.5, zorder=1)
    axp.text((bi + 0.5) * TS, 1.02, BLOCKS[bi]['name'].split('. ', 1)[1],
             ha='center', va='bottom', fontsize=6, color='#555555', transform=axp.get_xaxis_transform())
axp.axhline(0.5, color='#bbbbbb', lw=0.5, ls=':', zorder=1)
axp.fill_between(t, pA_lo[::100], pA_hi[::100], color=CH_A, alpha=0.18, lw=0, zorder=2)
axp.plot(t, pA_med[::100], color=CH_A, lw=1.0, zorder=3)
axp.set_ylim(0, 1); axp.set_xlim(0, TOTAL)
nature_ax(axp, 'timestep', 'P(choice a)', nx=6, ny=3)
axp.text(0.995, 0.06, 'median ± IQR', transform=axp.transAxes, ha='right', va='bottom',
         fontsize=6, color=CH_A)

fig.savefig(os.path.join(FIGDIR, 'F6_timeseries.svg'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'F6_timeseries.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ---- sample attractor coordinates per block (a few seeds, subsampled points) --------
# Use the second half of each block (settled onto the block's attractor). x = w2-w1, y = x2-x1,
# z = x0. Bandit block: decision steps only.
rng = np.random.default_rng(0)
def block_cloud(bi):
    lo, hi = bi * TS, (bi + 1) * TS
    half = lo + (hi - lo) // 2
    seeds = range(min(ATTR_MAX_SEEDS, N_SEEDS))
    xs, ys, zs = [], [], []
    for s in seeds:
        m = np.ones(hi - half, bool)
        if bandit[half:hi, s].any() and not bandit[half:hi, s].all():
            m = bandit[half:hi, s]
        ys.append(DW[half:hi, s][m]); xs.append(DX[half:hi, s][m]); zs.append(X0[half:hi, s][m])
    dW = np.concatenate(ys); dX = np.concatenate(xs); x0 = np.concatenate(zs)
    if dW.size > ATTR_N_SHOW:
        idx = rng.choice(dW.size, ATTR_N_SHOW, replace=False)
        dW, dX, x0 = dW[idx], dX[idx], x0[idx]
    return dict(dW=dW, dX=dX, x0=x0)

clouds = [block_cloud(bi) for bi in range(NB)]
BLOCK_COL = ['#967fb7', '#d76577', '#74ad83', '#EFA831', '#7991b5']   # one colour per block
# shared limits so the changing attractor is an honest comparison
_alldW = np.concatenate([c['dW'] for c in clouds]); _alldX = np.concatenate([c['dX'] for c in clouds])
_allx0 = np.concatenate([c['x0'] for c in clouds])
DWLIM = tuple(np.percentile(_alldW, [0.5, 99.5])); DXLIM = tuple(np.percentile(_alldX, [0.5, 99.5]))
X0LIM = (0, 1)#tuple(np.percentile(_allx0, [0.5, 99.5]))
print('cloud sizes:', [c['dW'].size for c in clouds])

In [ ]:
# ---- Figure C: 3D attractors, one panel per block  (x0, w2-w1, x2-x1) ---------------
# PM-attractor axes (matches F4_pm draw_clouds3d): x = x0, y = w2-w1, z = x2-x1.
def _style3d(ax):
    for pane in (ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane):
        pane.fill = False; pane.set_edgecolor('#CCCCCC'); pane.set_linewidth(0.25)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_major_locator(mticker.MaxNLocator(3)); axis.line.set_linewidth(0.25)
    ax.grid(False)
    ax.tick_params(labelsize=6, pad=-2, colors='#555555', width=0.25)
    ax.set_xlabel('$x_0$', fontsize=8, labelpad=-6)
    ax.set_ylabel('$w_2-w_1$', fontsize=8, labelpad=-6)
    ax.set_zlabel('$x_2-x_1$', fontsize=8, labelpad=-6)

fig = plt.figure(figsize=(1.5 * NB + 0.4, 1.8))
for bi, c in enumerate(clouds):
    ax = fig.add_subplot(1, NB, bi + 1, projection='3d')
    ax.scatter(c['x0'], c['dW'], c['dX'], s=2, c=BLOCK_COL[bi], alpha=0.25,
               linewidths=0, rasterized=True, depthshade=False)
    ax.set_xlim3d(*X0LIM); ax.set_ylim3d(*DWLIM); ax.set_zlim3d(*DXLIM)
    _style3d(ax); ax.view_init(elev=18, azim=-60); ax.set_box_aspect((1, 1, 0.85))
    ax.set_title(BLOCKS[bi]['name'].split('. ', 1)[1], fontsize=6.5, color=BLOCK_COL[bi], pad=-2)
fig.suptitle('Attractor per block (3D): x0, w2-w1, x2-x1', fontsize=8.5, y=1.06)
fig.subplots_adjust(left=0.02, right=0.98, bottom=0.06, top=0.9, wspace=0.12)
if SAVE:
    fig.savefig(os.path.join(FIGDIR, 'F6_attractors3d.svg'), dpi=300, bbox_inches='tight')
    fig.savefig(os.path.join(FIGDIR, 'F6_attractors3d.png'), dpi=300, bbox_inches='tight')
plt.show()
print('saved F6_attractors3d.svg' if SAVE else 'not saved')

In [ ]:
# ---- summary numbers ----------------------------------------------------------------
print(f"{'block':30s} {'P(a) start':>10} {'P(a) end':>9} {'verdict':>12} {'alpha':>6} {'p':>6}")
for bi, blk in enumerate(BLOCKS):
    lo, hi = bi * TS, (bi + 1) * TS
    ps = choice[lo:lo + 500].mean(); pe = choice[hi - 500:hi].mean()
    v, n, fr = block_fits[bi]
    tag = 'n/a' if np.isnan(v) else ('power law' if v > 0.5 else 'exponential')
    a = '' if fr is None else f'{fr.alpha:>6.2f}'
    p = '' if fr is None else f'{fr.loglik_ratio_p:>6.3f}'
    print(f"{blk['name']:30s} {ps:>10.2f} {pe:>9.2f} {tag:>12} {a:>6} {p:>6}")